## Feature engineering

In [212]:
import pandas as pd

In [213]:
df = pd.read_csv(
    "../data/preprocessed/ames_house_prices_cleaned.csv",
    keep_default_na=False,      # Don't auto-convert "None", "NA", etc. to NaN
    na_values=['']              # Only treat truly empty strings as null
)

In [214]:
pd.set_option("display.max_columns", None)

#### dividing columns into numerical and categorical cols

In [215]:
numeric_cols = df.select_dtypes(include=["Float64", "int64"])
numeric_cols.shape[1] # <-- total numerical cols

38

In [216]:
categorical_cols = df.select_dtypes(include=["str"])
categorical_cols.shape[1] # <--- total categorical cols  

39

#### seeing skewness in the distribution

SalePrice distribution is highly right skewed because skewness value is greater than +1

In [217]:
df["SalePrice"].skew()  

np.float64(1.8828757597682129)

strong candidate features that most influence the price

In [218]:
candidate_features = [
    "OverallQual",
    "Neighborhood",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "YearBuilt",
    "KitchenQual",
    "FullBath",
    "LotArea",
]

identifying ordinal features

In [219]:
ordinal_features = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'KitchenQual',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'HeatingQC',
    'BsmtExposure',
]

listing Nominal feature no natural order

In [220]:
nominal_features = list(
    df.drop(columns=ordinal_features + list(numeric_cols.columns)).columns
)

**features engineering from existing ones**
- total bathrooms in house
- age of house
- garageAge 
- Total living Area

In [221]:
df['TotalBathrooms'] = df["FullBath"] + (0.5*df["HalfBath"]) + df['BsmtFullBath'] + (0.5*df['BsmtHalfBath'])  # Multiplying half bath with 0.5

In [222]:
df['HouseAge'] = df['YrSold'] - df['YearBuilt']

In [223]:
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']

In [224]:
df['GarageAge']  = df['YrSold'] - df['GarageYrBlt']

In [225]:
df['TotalLivingArea'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

dropping Id column as it doesn't help in any prediction work and also Segment i column i don't want model to learn this pattern

In [226]:
df = df.drop(
    columns=["Id", "SaleType", "SaleCondition", "Segment"]
)

saving new feature engineered dataset

In [ ]:
# df.to_csv(r'../data/final/housing_feature_engineered.csv', index=False)

#### Ordinal Encoding
quality mapping for ordinal features

In [228]:
# # Define quality mapping for ordinal features
# quality_mapping = {
#     'None': 0,  # No feature present
#     'Po': 1,    # Poor
#     'Fa': 2,    # Fair
#     'TA': 3,    # Average/Typical
#     'Gd': 4,    # Good
#     'Ex': 5     # Excellent
# }

# # Special mapping for BsmtExposure
# basement_exposure_mapping = {
#     'None': 0,  # No basement
#     'No': 1,    # No exposure
#     'Mn': 2,    # Minimum exposure
#     'Av': 3,    # Average exposure
#     'Gd': 4     # Good exposure
# }

# # Apply quality mapping to all ordinal features except BsmtExposure
# quality_cols = [col for col in ordinal_features if col != 'BsmtExposure']
# for col in quality_cols:
#     if col in df.columns:
#         df[col] = df[col].map(quality_mapping)

# # Apply basement exposure mapping
# if 'BsmtExposure' in df.columns:
#     df['BsmtExposure'] = df['BsmtExposure'].map(basement_exposure_mapping)

#### One-Hot Encoding

In [229]:
# Apply one-hot encoding to all nominal features
# df = pd.get_dummies(df, columns=nominal_features, drop_first=False)